# Debug InternVL3 Shape Mismatch Issue

This notebook investigates the shape mismatch error occurring during InternVL3 evaluation:
```
ERROR: shape mismatch: value tensor of shape [256, 896] cannot be broadcast to indexing result of shape [1, 896]
```

The error occurs when LatentWrapper delegates to the base model during generation, specifically when there are no latent spans detected.

In [ ]:
import torch
import sys
import os
sys.path.append('/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco')

from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
import logging

# Set up logging to see debug messages
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name())

## Section 1: Load and Inspect Model Configuration

Let's load the InternVL3-1B-Pretrained model and examine its structure to understand how it processes vision and text inputs.

In [ ]:
# Load the InternVL3 model
model_name = "OpenGVLab/InternVL3-1B-Pretrained"

print("Loading model and tokenizer...")
try:
    model = AutoModel.from_pretrained(
        model_name, 
        trust_remote_code=True, 
        torch_dtype=torch.bfloat16, 
        low_cpu_mem_usage=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
    image_processor = AutoImageProcessor.from_pretrained(model_name, trust_remote_code=True)
    
    print("✓ Model loaded successfully")
    print(f"Model type: {type(model)}")
    print(f"Model dtype: {model.dtype}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

In [ ]:
# Examine model structure
print("Model components:")
for name, module in model.named_children():
    print(f"  {name}: {type(module)}")

print("\nModel configuration:")
print(f"  Vision config: {model.config.vision_config}")
print(f"  LLM config keys: {list(model.config.llm_config.keys())}")
print(f"  Vision hidden size: {model.config.vision_config.hidden_size}")
print(f"  LLM hidden size: {model.config.llm_config.hidden_size}")
print(f"  Image size: {model.config.vision_config.image_size}")
print(f"  Patch size: {model.config.vision_config.patch_size}")
print(f"  Num image tokens: {model.num_image_token}")

# Examine the mlp1 projector
print(f"\nMLF1 (projector) structure: {model.mlp1}")

# Check if vision_model and language_model exist
print(f"\nVision model exists: {hasattr(model, 'vision_model')}")
print(f"Language model exists: {hasattr(model, 'language_model')}")

if hasattr(model, 'vision_model'):
    print(f"Vision model type: {type(model.vision_model)}")
if hasattr(model, 'language_model'):
    print(f"Language model type: {type(model.language_model)}")

## Section 2: Check Input and Output Tensor Shapes

Let's create sample inputs matching the error scenario and examine their shapes through the model pipeline.

In [ ]:
# Create sample inputs matching the error scenario
batch_size = 1  # This was 16 in the original error, but we reduced to 1
seq_length = 78  # From error log: torch.Size([1, 78])
image_size = 448  # From config

# Sample input_ids (text tokens)
input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, seq_length))
print(f"input_ids shape: {input_ids.shape}")

# Sample pixel_values (image)
pixel_values = torch.randn(batch_size, 3, image_size, image_size, dtype=torch.bfloat16)
print(f"pixel_values shape: {pixel_values.shape}")
print(f"pixel_values dtype: {pixel_values.dtype}")

# Sample attention mask
attention_mask = torch.ones(batch_size, seq_length)
print(f"attention_mask shape: {attention_mask.shape}")

# Check model's expected device and move tensors accordingly
model_device = next(model.parameters()).device
model_dtype = next(model.parameters()).dtype

print(f"Model device: {model_device}")
print(f"Model dtype: {model_dtype}")

# Move tensors to model device and dtype
input_ids = input_ids.to(device=model_device)
pixel_values = pixel_values.to(device=model_device, dtype=model_dtype)
attention_mask = attention_mask.to(device=model_device)

print("✓ Tensors moved to model device and dtype")

## Section 3: Run Forward Pass with Sample Data

Let's examine how the vision model processes inputs and where the shape mismatch might occur.

In [ ]:
# Step-by-step forward pass analysis
print("=== Vision Model Processing ===")

with torch.no_grad():
    # Process through vision model
    try:
        vision_outputs = model.vision_model(pixel_values)
        print(f"✓ Vision model output shape: {vision_outputs.last_hidden_state.shape}")
        print(f"  Vision model output type: {type(vision_outputs)}")
        
        # Process through projector (mlp1)
        projected_vision = model.mlp1(vision_outputs.last_hidden_state)
        print(f"✓ Projected vision shape: {projected_vision.shape}")
        
        # Expected shape is [batch_size, num_patches, hidden_size]
        # Where num_patches should be around 256 for 448x448 image with 14x14 patches
        # 448/14 = 32, so 32*32 = 1024 patches, but there's also a CLS token = 1025
        
    except Exception as e:
        print(f"❌ Error in vision processing: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# Test full model forward pass
print("\n=== Full Model Forward Pass ===")

try:
    with torch.no_grad():
        # Try forward pass
        outputs = model.forward(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        print(f"✓ Model forward pass successful")
        print(f"  Output logits shape: {outputs.logits.shape}")
        print(f"  Expected: [batch_size, seq_length, vocab_size] = [{batch_size}, {seq_length}, {tokenizer.vocab_size}]")
        
except Exception as e:
    print(f"❌ Error in full forward pass: {e}")
    import traceback
    traceback.print_exc()

## Section 4: Debug LatentWrapper.generate Method

Now let's test the specific case that's failing - the model's generate method, which is where the error occurs.

In [ ]:
# Test the generate method directly (this is where the error occurs)
print("=== Testing Model Generate Method ===")

generation_config = {
    'max_new_tokens': 10,  # Reduced for testing
    'do_sample': True,
    'num_beams': 1,
    'temperature': 0.8,
    'top_p': 0.9,
    'top_k': 50
}

try:
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            **generation_config
        )
        
        print(f"✓ Generation successful!")
        print(f"  Generated ids shape: {generated_ids.shape}")
        print(f"  Generated text: {tokenizer.decode(generated_ids[0], skip_special_tokens=True)}")
        
except Exception as e:
    print(f"❌ Error in generation: {e}")
    print(f"Error type: {type(e)}")
    import traceback
    traceback.print_exc()
    
    # Let's examine the error more closely
    if "shape mismatch" in str(e):
        print(f"\n🔍 Shape mismatch detected!")
        print(f"Error message: {str(e)}")
        
        # Try to identify which shapes are mismatched
        if "[256, 896]" in str(e):
            print("  - 256 likely refers to num_image_tokens (256)")
            print("  - 896 is the language model hidden size")
        if "[1, 896]" in str(e):
            print("  - 1 is the batch size")
            print("  - 896 is the language model hidden size")

In [ ]:
# Let's check if the issue is related to how we format multimodal prompts
print("\n=== Investigating Multimodal Prompt Format ===")

# Check what special tokens exist
print("Special tokens:")
for token in ['<img>', '</img>', '<IMG_CONTEXT>']:
    if token in tokenizer.get_vocab():
        token_id = tokenizer.convert_tokens_to_ids(token)
        print(f"  {token}: {token_id}")
    else:
        print(f"  {token}: NOT FOUND")

# Check num_image_token
print(f"Model num_image_token: {model.num_image_token}")
print(f"Model img_context_token_id: {model.img_context_token_id}")

# Let's try creating a proper multimodal prompt
test_prompt = "Describe this image."
img_context = "<IMG_CONTEXT>" * model.num_image_token
multimodal_prompt = f"<img>{img_context}</img>{test_prompt}"

print(f"Multimodal prompt length: {len(multimodal_prompt)}")
print(f"Sample: {multimodal_prompt[:100]}...")

# Tokenize the multimodal prompt
try:
    tokenized = tokenizer(multimodal_prompt, return_tensors="pt")
    test_input_ids = tokenized.input_ids.to(model_device)
    test_attention_mask = tokenized.attention_mask.to(model_device)
    
    print(f"Tokenized input shape: {test_input_ids.shape}")
    print(f"Number of IMG_CONTEXT tokens: {(test_input_ids == model.img_context_token_id).sum().item()}")
    
except Exception as e:
    print(f"❌ Error tokenizing multimodal prompt: {e}")

In [ ]:
# Test generation with properly formatted multimodal prompt
print("\n=== Testing Generation with Proper Multimodal Prompt ===")

try:
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=test_input_ids,
            attention_mask=test_attention_mask,
            pixel_values=pixel_values,
            **generation_config
        )
        
        print(f"✓ Generation with multimodal prompt successful!")
        print(f"  Generated ids shape: {generated_ids.shape}")
        print(f"  Generated text: {tokenizer.decode(generated_ids[0], skip_special_tokens=True)}")
        
except Exception as e:
    print(f"❌ Error in multimodal generation: {e}")
    import traceback
    traceback.print_exc()
    
    # Additional debugging for the shape mismatch
    print(f"\n🔍 Additional Error Analysis:")
    print(f"  Input ids shape: {test_input_ids.shape}")
    print(f"  Pixel values shape: {pixel_values.shape}")
    print(f"  Expected num_image_tokens: {model.num_image_token}")
    
    # Check if this is related to image token processing
    num_img_tokens_in_prompt = (test_input_ids == model.img_context_token_id).sum().item()
    print(f"  Actual IMG_CONTEXT tokens in prompt: {num_img_tokens_in_prompt}")
    
    if num_img_tokens_in_prompt != model.num_image_token:
        print(f"  ⚠️  Mismatch! Expected {model.num_image_token}, got {num_img_tokens_in_prompt}")

## Section 5: Test Base Model Delegation and Error Handling

Let's examine how the LatentWrapper processes inputs and where the delegation fails.

In [ ]:
# Load and test LatentWrapper 
print("=== Testing LatentWrapper ===")

try:
    from multicoco.latent_wrapper import LatentWrapper
    
    # Create latent wrapper
    wrapped_model = LatentWrapper(model, tokenizer)
    print("✓ LatentWrapper created successfully")
    
    # Test if it has latent spans (should be False for our test input)
    has_latent = wrapped_model._has_latent_spans(test_input_ids)
    print(f"Has latent spans: {has_latent}")
    
    # Try the generate method
    try:
        with torch.no_grad():
            generated_ids = wrapped_model.generate(
                input_ids=test_input_ids,
                attention_mask=test_attention_mask,
                pixel_values=pixel_values,
                **generation_config
            )
            
            print(f"✓ LatentWrapper generation successful!")
            print(f"  Generated ids shape: {generated_ids.shape}")
            
    except Exception as e:
        print(f"❌ Error in LatentWrapper generation: {e}")
        
        # This is where our shape mismatch occurs
        print(f"\n🔍 LatentWrapper Error Analysis:")
        print(f"  Error type: {type(e)}")
        print(f"  Error message: {str(e)}")
        
        # Check if it's the expected shape mismatch
        if "shape mismatch" in str(e) and "256, 896" in str(e):
            print(f"  ✓ Confirmed: This is the shape mismatch we're investigating!")
            print(f"    - [256, 896] suggests vision token embeddings")
            print(f"    - [1, 896] suggests batch of language embeddings")
            
        import traceback
        traceback.print_exc()
        
except ImportError as e:
    print(f"❌ Error importing LatentWrapper: {e}")
except Exception as e:
    print(f"❌ Error creating LatentWrapper: {e}")
    import traceback
    traceback.print_exc()

## Section 6: Validate Shape Broadcasting Logic

Let's investigate the specific shape mismatch and understand what's happening during tensor operations.

In [ ]:
# Test shape broadcasting scenarios
print("=== Shape Broadcasting Analysis ===")

# Create tensors with the problematic shapes
shape_256_896 = torch.randn(256, 896)  # Vision token embeddings  
shape_1_896 = torch.randn(1, 896)      # Batch of language embeddings

print(f"Tensor A shape: {shape_256_896.shape}")
print(f"Tensor B shape: {shape_1_896.shape}")

# Test different operations to understand the broadcasting issue
print("\n1. Broadcasting compatibility test:")
try:
    result = shape_256_896 + shape_1_896
    print(f"  ✓ Addition works: {result.shape}")
except Exception as e:
    print(f"  ❌ Addition fails: {e}")

print("\n2. Assignment/indexing test:")
try:
    # This might be what's failing - trying to assign 256x896 to 1x896 slot
    target = torch.zeros(1, 896)
    target[:] = shape_256_896  # This should fail
    print(f"  ✓ Assignment works")
except Exception as e:
    print(f"  ❌ Assignment fails: {e}")
    print(f"    This might be the source of our error!")

print("\n3. Index assignment test:")
try:
    # Testing what happens when we try to index assign
    target = torch.zeros(10, 896)
    target[0] = shape_1_896.squeeze(0)  # This should work
    print(f"  ✓ Index assignment works with proper reshaping")
    
    target[0] = shape_256_896  # This should fail
    print(f"  ✓ Direct assignment somehow worked (unexpected)")
except Exception as e:
    print(f"  ❌ Index assignment fails: {e}")

print("\n4. Possible solutions:")
print(f"  - Reshape [256, 896] to [1, 256*896] = {shape_256_896.view(1, -1).shape}")
print(f"  - Take first token: [256, 896] -> [1, 896] = {shape_256_896[0:1].shape}")
print(f"  - Pool tokens: [256, 896] -> [1, 896] = {shape_256_896.mean(dim=0, keepdim=True).shape}")

## Root Cause Investigation

Based on the analysis above, the issue seems to be that InternVL3 is trying to process vision tokens during generation, but there's a mismatch between:

1. **Vision embeddings**: Shape `[256, 896]` - These are the 256 image tokens projected to 896-dim language space
2. **Expected input**: Shape `[1, 896]` - The model expects embeddings for a single sequence position

This suggests the issue is in how InternVL3 handles the transition between vision tokens and language tokens during generation. The model is trying to assign or index vision token embeddings into a language token slot, causing the broadcasting error.

**Potential Solutions:**
1. Fix how vision tokens are processed during generation
2. Ensure proper padding/masking of vision tokens  
3. Check if the multimodal prompt format is correct
4. Investigate InternVL3's specific requirements for generation vs. forward pass

## SOLUTION FOUND!

After analyzing the InternVL source code, I found the exact cause of the error. The issue is in `modeling_internvl_chat.py` line 177-186:

```python
input_embeds[selected] = input_embeds[selected] * 0.0 + vit_embeds.reshape(-1, C)
```

The problem is:
1. **Vision embeddings**: After processing through `extract_feature()`, they have a different number of tokens than expected
2. **IMG_CONTEXT tokens**: The prompt contains 256 `<IMG_CONTEXT>` tokens 
3. **Mismatch**: The processed vision embeddings don't match the number of IMG_CONTEXT tokens

The error occurs because:
- `input_embeds[selected]` has shape `[256, 896]` (256 IMG_CONTEXT tokens)
- `vit_embeds.reshape(-1, C)` has a different first dimension

**Root Cause**: The issue is NOT in LatentWrapper but in how the multimodal prompt is formatted. The number of `<IMG_CONTEXT>` tokens must exactly match the number of vision tokens produced by `extract_feature()`.